# NYC Neighborhood Pulse
## Notebook 03 — Visualization Prototype (PyDeck)

**No GeoJSON needed!** Uses listing coordinates directly.

- **Map A**: ScatterplotLayer — topic bubbles per neighborhood  
- **Map B**: HexagonLayer — 3D listing density heatmap  
- **Map C**: Combined (best, use this in Streamlit)

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import pydeck as pdk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

# Neighborhood topic summary (from notebook 02)
neigh_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'neighborhoods_topics.csv'))

# Raw listings — for HexagonLayer density map
listings = pd.read_csv(
    os.path.join(BASE_DIR, 'listings.csv'),
    usecols=['latitude', 'longitude']
).dropna().rename(columns={'latitude': 'lat', 'longitude': 'lon'})

print(f'Neighborhoods : {len(neigh_df)}')
print(f'Listings (hex): {len(listings):,}')
neigh_df.head(3)

## 2. Color Palette

In [ ]:
TOPIC_COLORS = {
    'Transit & Commute':          [0,   200, 255, 220],
    'Food & Restaurants':         [255, 100, 0,   220],
    'Safety & Neighborhood Vibe': [220, 50,  255, 220],
    'Parks & Outdoor Life':       [50,  255, 100, 220],
    'Nightlife & Entertainment':  [255, 50,  100, 220],
    'Arts & Culture':             [180, 0,   255, 220],
    'Shopping & Retail':          [255, 220, 0,   220],
    'Families & Community':       [100, 200, 255, 220],
    'Luxury & Upscale':           [255, 160, 50,  220],
    'Diversity & Local Character':[50,  255, 200, 220],
    'Other':                      [150, 150, 150, 150],
    'Unclassified':               [80,  80,  80,  100],
}
DEFAULT_COLOR = [100, 100, 100, 150]

neigh_df['color'] = neigh_df['dominant_topic_label'].apply(
    lambda t: TOPIC_COLORS.get(t, DEFAULT_COLOR)
)
neigh_df['radius'] = (
    np.log1p(neigh_df['review_count'])
    / np.log1p(neigh_df['review_count'].max())
    * 600 + 150
)

print('Topics:', neigh_df['dominant_topic_label'].unique().tolist())

## 3. Map A — Topic Scatter (neighborhood bubbles)

In [ ]:
scatter_layer = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=80,
    get_fill_color='color',
    get_line_color=[255, 255, 255, 60],
    line_width_min_pixels=1,
    opacity=0.85,
)

view_state = pdk.ViewState(
    latitude=40.7128, longitude=-74.006,
    zoom=10.5, pitch=0
)

tooltip = {
    'html': '''
        <div style="background:rgba(0,0,0,0.88);padding:12px 16px;
                    border-radius:8px;font-family:sans-serif;color:white;max-width:240px;">
          <b style="font-size:14px;">{neighborhood}</b>
          <span style="color:#888;font-size:11px;"> · {borough}</span><br/><br/>
          <span style="color:#aaa;font-size:11px;">Dominant Topic</span><br/>
          <span style="font-size:13px;color:#ff80ff;">{dominant_topic_label}</span><br/><br/>
          <span style="color:#aaa;font-size:11px;">Keywords: </span>
          <span style="font-size:11px;">{top_keywords}</span><br/>
          <span style="color:#aaa;font-size:11px;">Reviews: {review_count}</span>
        </div>
    ''',
    'style': {'backgroundColor': 'transparent'}
}

deck_a = pdk.Deck(
    layers=[scatter_layer],
    initial_view_state=view_state,
    map_style='mapbox://styles/mapbox/dark-v10',
    tooltip=tooltip,
)
out_a = os.path.join(OUTPUT_DIR, 'map_A_scatter.html')
deck_a.to_html(out_a, open_browser=True)
print(f'✅ Map A saved → {out_a}')

## 4. Map B — HexagonLayer 3D Density (all listings)

In [ ]:
hex_layer = pdk.Layer(
    'HexagonLayer',
    data=listings,
    get_position='[lon, lat]',
    radius=300,
    elevation_scale=6,
    elevation_range=[0, 800],
    extruded=True,
    pickable=True,
    color_range=[           # viridis-style: purple → pink → orange → yellow
        [68,  1,   84,  200],
        [114, 0,   168, 200],
        [188, 0,   167, 200],
        [248, 81,  106, 200],
        [254, 168, 60,  200],
        [254, 231, 37,  200],
    ],
    coverage=0.9,
)

view_3d = pdk.ViewState(
    latitude=40.7128, longitude=-74.006,
    zoom=10.5, pitch=45, bearing=-10
)

deck_b = pdk.Deck(
    layers=[hex_layer],
    initial_view_state=view_3d,
    map_style='mapbox://styles/mapbox/dark-v10',
    tooltip={'text': 'Listings: {elevationValue}'},
)
out_b = os.path.join(OUTPUT_DIR, 'map_B_hexagon.html')
deck_b.to_html(out_b, open_browser=True)
print(f'✅ Map B saved → {out_b}')

## 5. Map C — Combined (flat hex background + topic circles) ⭐ BEST

In [ ]:
hex_bg = pdk.Layer(
    'HexagonLayer',
    data=listings,
    get_position='[lon, lat]',
    radius=250,
    extruded=False,           # flat background
    pickable=False,
    color_range=[
        [20,  10,  40,  160],
        [80,  10,  120, 160],
        [160, 20,  160, 160],
        [220, 60,  120, 160],
        [255, 120, 50,  160],
        [255, 220, 30,  160],
    ],
    coverage=0.85,
    opacity=0.7,
)

scatter_top = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=50,
    get_fill_color='color',
    get_line_color=[255, 255, 255, 80],
    line_width_min_pixels=1,
    opacity=0.9,
)

deck_c = pdk.Deck(
    layers=[hex_bg, scatter_top],
    initial_view_state=view_state,
    map_style='mapbox://styles/mapbox/dark-v10',
    tooltip=tooltip,
)
out_c = os.path.join(OUTPUT_DIR, 'map_C_combined.html')
deck_c.to_html(out_c, open_browser=True)
print(f'✅ Map C (combined) saved → {out_c}')
print('   This is the one to use in Streamlit!')

## 6. Legend

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor('#111111')
fig.patch.set_facecolor('#111111')
ax.axis('off')
ax.set_title('Topic Color Legend', color='white', fontsize=13, pad=10)

patches = [
    mpatches.Patch(color=[c/255 for c in rgba[:3]] + [0.9], label=label)
    for label, rgba in TOPIC_COLORS.items()
    if label not in ('Unclassified', 'Other')
]
ax.legend(handles=patches, loc='center', frameon=False,
          labelcolor='white', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'legend.png'), dpi=150, facecolor='#111111')
plt.show()
print('Legend saved.')

---
## ✅ Done! Three maps in `outputs/`

| File | Style |
|---|---|
| `map_A_scatter.html` | Topic color bubbles |
| `map_B_hexagon.html` | 3D density heatmap |
| `map_C_combined.html` | ⭐ Both combined — use in Streamlit |

**Next**: `streamlit run app.py`

In [ ]:
# 快速修复：换成 Carto 免费暗色底图（不需要 Mapbox token）
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'

deck_a.map_style = CARTO_DARK
deck_a.to_html(out_a)

deck_b.map_style = CARTO_DARK
deck_b.to_html(out_b)

deck_c.map_style = CARTO_DARK
deck_c.to_html(out_c, open_browser=True)

print('✅ 三张地图已更新为 Carto 暗色底图，刷新浏览器查看！')

In [ ]:
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'

# 重建 Map A
scatter_layer2 = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=80,
    get_fill_color='color',
    get_line_color=[255, 255, 255, 80],
    line_width_min_pixels=1,
    opacity=0.9,
)

deck_a2 = pdk.Deck(
    layers=[scatter_layer2],
    initial_view_state=view_state,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
deck_a2.to_html(out_a, open_browser=True)
print(f'✅ Map A rebuilt → {out_a}')

# 重建 Map C
hex_bg2 = pdk.Layer(
    'HexagonLayer',
    data=listings,
    get_position='[lon, lat]',
    radius=250,
    extruded=False,
    pickable=False,
    color_range=[
        [20,  10,  40,  160],
        [80,  10,  120, 160],
        [160, 20,  160, 160],
        [220, 60,  120, 160],
        [255, 120, 50,  160],
        [255, 220, 30,  160],
    ],
    coverage=0.85,
    opacity=0.7,
)

scatter_top2 = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=50,
    get_fill_color='color',
    get_line_color=[255, 255, 255, 80],
    line_width_min_pixels=1,
    opacity=0.9,
)

deck_c2 = pdk.Deck(
    layers=[hex_bg2, scatter_top2],
    initial_view_state=view_state,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
deck_c2.to_html(out_c, open_browser=True)
print(f'✅ Map C rebuilt → {out_c}')

In [ ]:
print(neigh_df[['neighborhood', 'dominant_topic_label', 'color', 'radius']].head(10))
print(f'\nRadius range: {neigh_df["radius"].min():.0f} ~ {neigh_df["radius"].max():.0f}')
print(f'Unique topics: {neigh_df["dominant_topic_label"].nunique()}')

In [ ]:
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'

# 修复后的 scatter：radius_scale 改成 1（单位就是米，194~750米正合适）
scatter_fixed = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=1,          # ← 关键修复，之前80是错的
    get_fill_color='color',
    get_line_color=[255, 255, 255, 80],
    line_width_min_pixels=1,
    opacity=0.9,
)

deck_a_fixed = pdk.Deck(
    layers=[scatter_fixed],
    initial_view_state=view_state,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
deck_a_fixed.to_html(out_a, open_browser=True)
print('✅ Map A fixed!')

# Map C 也一起修
deck_c_fixed = pdk.Deck(
    layers=[hex_bg2, scatter_fixed],
    initial_view_state=view_state,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
deck_c_fixed.to_html(out_c, open_browser=True)
print('✅ Map C fixed!')

In [ ]:
##

In [ ]:
import pandas as pd
import numpy as np
import pydeck as pdk
import os

BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'

# 重新加载更新后的数据
neigh_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'neighborhoods_topics.csv'))

TOPIC_COLORS = {
    'Spanish-Speaking Visitors':  [255, 80,  80,  220],   # red
    'Clean & Comfortable Stay':   [100, 200, 255, 220],   # sky blue
    'Great Location & Transit':   [0,   200, 255, 220],   # cyan
    'Truly Enjoyable Experience': [50,  255, 100, 220],   # green
    'Feels Like Home':            [255, 160, 50,  220],   # gold
    'Practical Comfort':          [180, 0,   255, 220],   # purple
    'Nice & Pleasant Place':      [255, 220, 0,   220],   # yellow
    'German-Speaking Visitors':   [255, 50,  150, 220],   # pink
    'French-Speaking Visitors':   [50,  255, 200, 220],   # teal
    'NYC City Experience':        [220, 50,  255, 220],   # magenta
}
DEFAULT_COLOR = [100, 100, 100, 150]

neigh_df['color'] = neigh_df['dominant_topic_label'].apply(
    lambda t: TOPIC_COLORS.get(t, DEFAULT_COLOR)
)
neigh_df['radius'] = (
    np.log1p(neigh_df['review_count'])
    / np.log1p(neigh_df['review_count'].max()) * 600 + 150
)

print('Topic distribution in map:')
print(neigh_df['dominant_topic_label'].value_counts())

In [ ]:
# 加载 listings 做 hex 底图
listings = pd.read_csv(
    os.path.join(BASE_DIR, 'listings.csv'),
    usecols=['latitude', 'longitude']
).dropna().rename(columns={'latitude': 'lat', 'longitude': 'lon'})

view_state = pdk.ViewState(
    latitude=40.7128, longitude=-74.006, zoom=10.5, pitch=0
)

tooltip = {
    'html': '''
        <div style="background:rgba(0,0,0,0.88);padding:12px 16px;
                    border-radius:8px;font-family:sans-serif;color:white;max-width:240px;">
          <b style="font-size:14px;">{neighborhood}</b>
          <span style="color:#888;font-size:11px;"> · {borough}</span><br/><br/>
          <span style="color:#aaa;font-size:11px;">Dominant Topic</span><br/>
          <span style="font-size:13px;color:#ff80ff;">{dominant_topic_label}</span><br/><br/>
          <span style="color:#aaa;font-size:11px;">Reviews: {review_count}</span>
        </div>''',
    'style': {'backgroundColor': 'transparent'}
}

hex_bg = pdk.Layer('HexagonLayer', data=listings,
    get_position='[lon, lat]', radius=250, extruded=False,
    pickable=False, opacity=0.7, coverage=0.85,
    color_range=[
        [20,10,40,160],[80,10,120,160],[160,20,160,160],
        [220,60,120,160],[255,120,50,160],[255,220,30,160],
    ]
)

scatter = pdk.Layer('ScatterplotLayer', data=neigh_df,
    pickable=True, get_position='[lon, lat]',
    get_radius='radius', radius_scale=1,
    get_fill_color='color',
    get_line_color=[255,255,255,80],
    line_width_min_pixels=1, opacity=0.9,
)

deck = pdk.Deck(
    layers=[hex_bg, scatter],
    initial_view_state=view_state,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
out = os.path.join(OUTPUT_DIR, 'map_C_combined.html')
deck.to_html(out, open_browser=True)
print(f'✅ Map exported → {out}')

In [ ]:
view_3d = pdk.ViewState(
    latitude=40.7128, longitude=-74.006,
    zoom=10.5, pitch=45, bearing=-10
)
hex_layer = pdk.Layer('HexagonLayer', data=listings,
    get_position='[lon, lat]', radius=300,
    elevation_scale=6, elevation_range=[0, 800],
    extruded=True, pickable=True,
    color_range=[
        [68,1,84,200],[114,0,168,200],[188,0,167,200],
        [248,81,106,200],[254,168,60,200],[254,231,37,200],
    ], coverage=0.9,
)
pdk.Deck(
    layers=[hex_layer], initial_view_state=view_3d,
    map_style=CARTO_DARK,
    tooltip={'text': 'Listings: {elevationValue}'},
).to_html(os.path.join(OUTPUT_DIR, 'map_B_hexagon.html'), open_browser=True)
print('✅ Hex map regenerated!')

In [ ]:
import pandas as pd
import numpy as np
import pydeck as pdk
import os

BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
CARTO_DARK = 'https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json'

neigh_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'neighborhoods_topics.csv'))

TOPIC_COLORS = {
    'Spanish-Speaking Visitors':  [255, 80,  80,  200],
    'Clean & Comfortable Stay':   [100, 200, 255, 200],
    'Great Location & Transit':   [0,   220, 255, 200],
    'Truly Enjoyable Experience': [50,  255, 100, 200],
    'Feels Like Home':            [255, 160, 50,  200],
    'Practical Comfort':          [180, 0,   255, 200],
    'Nice & Pleasant Place':      [255, 220, 0,   200],
    'German-Speaking Visitors':   [255, 50,  150, 200],
    'French-Speaking Visitors':   [50,  255, 200, 200],
    'NYC City Experience':        [220, 50,  255, 200],
}

neigh_color_map = neigh_df.set_index('neighbourhood_cleansed' if 'neighbourhood_cleansed' in neigh_df.columns else 'neighborhood')['dominant_topic_label'].to_dict()
neigh_df_map = neigh_df.set_index('neighborhood')[['dominant_topic_label']].to_dict('index')

# 加载每个 listing，带上社区名
listings_full = pd.read_csv(
    os.path.join(BASE_DIR, 'listings.csv'),
    usecols=['latitude', 'longitude', 'neighbourhood_cleansed']
).dropna().rename(columns={
    'latitude': 'lat',
    'longitude': 'lon',
    'neighbourhood_cleansed': 'neighborhood'
})

# 把 topic 颜色 join 到每个 listing
listings_full = listings_full.merge(
    neigh_df[['neighborhood', 'dominant_topic_label']],
    on='neighborhood', how='left'
)
listings_full['color'] = listings_full['dominant_topic_label'].apply(
    lambda t: TOPIC_COLORS.get(str(t), [120, 120, 120, 150])
)

print(f'Listings with topic color: {len(listings_full):,}')
print(listings_full['dominant_topic_label'].value_counts())

In [ ]:
# 艺术版终极地图

# 底层：密度热力（magma 色系）
heatmap_art = pdk.Layer(
    'HeatmapLayer',
    data=listings_full2,
    get_position='[lon, lat]',
    get_weight=1,
    radiusPixels=18,        # 紧凑，保留局部热点结构
    intensity=3,
    threshold=0.005,        # 连稀疏区域也显示
    color_range=[
        [10,   0,  20,  255],   # 近黑（极低密度）
        [80,   0,  120, 255],   # 深紫
        [180,  20, 120, 255],   # 品红
        [240,  80, 40,  255],   # 橙红
        [250,  180, 20, 255],   # 亮橙
        [252,  253, 191,255],   # 淡黄白（极高密度）
    ],
)

# 上层：每个 listing 一个发光点（半透明叠加产生辉光）
glow_layer = pdk.Layer(
    'ScatterplotLayer',
    data=listings_full2,
    get_position='[lon, lat]',
    get_radius=25,           # 25米，极小
    get_fill_color=[255, 220, 180, 18],  # 暖白，透明度极低
    stroked=False,
    opacity=1.0,
)

# 全纽约视角 — 城市轮廓本身就是艺术
view_nyc = pdk.ViewState(
    latitude=40.693,
    longitude=-73.960,
    zoom=11,
    pitch=0,
    bearing=0,
)

deck_art = pdk.Deck(
    layers=[heatmap_art, glow_layer],
    initial_view_state=view_nyc,
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-nolabels-gl-style/style.json',  # 无标签底图，更纯粹
)

out_art = os.path.join(OUTPUT_DIR, 'map_ART.html')
deck_art.to_html(out_art, open_browser=True)
print('✅ 艺术版地图生成完成！')

In [ ]:
# 双层叠加：热力底图 + topic 气泡
scatter_topic = pdk.Layer(
    'ScatterplotLayer',
    data=neigh_df,
    pickable=True,
    get_position='[lon, lat]',
    get_radius='radius',
    radius_scale=1,
    get_fill_color='color',
    get_line_color=[255, 255, 255, 60],
    line_width_min_pixels=1,
    opacity=0.85,
)

deck_combined_final = pdk.Deck(
    layers=[heatmap_layer, scatter_topic],
    initial_view_state=view_state3,
    map_style=CARTO_DARK,
    tooltip=tooltip,
)
out_cf = os.path.join(OUTPUT_DIR, 'map_G_final.html')
deck_combined_final.to_html(out_cf, open_browser=True)
print('✅ Combined final map exported!')

In [ ]:
# 先安装
!pip install h3

In [ ]:
import h3
import numpy as np
import matplotlib.cm as cm
import json

RESOLUTION = 11   # ~25米格子，接近建筑尺度

listings_h3 = pd.read_csv(
    os.path.join(BASE_DIR, 'listings.csv'),
    usecols=['latitude','longitude']
).dropna().rename(columns={'latitude':'lat','longitude':'lon'})

# 每个 listing → H3 格子编码
listings_h3['cell'] = listings_h3.apply(
    lambda r: h3.latlng_to_cell(r['lat'], r['lon'], RESOLUTION), axis=1
)

# 每个格子的 listing 数量
cell_counts = listings_h3.groupby('cell').size().reset_index(name='count')
print(f'Total H3 cells: {len(cell_counts):,}')

# 密度 → magma 颜色
vals = cell_counts['count'].values
norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-8)
cmap = cm.get_cmap('magma')
rgba = (cmap(norm) * 255).astype(int)
rgba[:, 3] = 230
cell_counts['color'] = rgba[:, :4].tolist()

# H3 格子 → GeoJSON polygon
def cell_to_feature(row):
    boundary = h3.cell_to_boundary(row['cell'])
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    return {
        'type': 'Feature',
        'geometry': {'type': 'Polygon', 'coordinates': [coords]},
        'properties': {'count': int(row['count']), 'color': row['color']}
    }

features = [cell_to_feature(r) for _, r in cell_counts.iterrows()]
geojson_h3 = {'type': 'FeatureCollection', 'features': features}
print(f'GeoJSON ready: {len(features)} polygons')

In [ ]:
h3_layer = pdk.Layer(
    'GeoJsonLayer',
    data=geojson_h3,
    pickable=False,
    stroked=False,          # 无边框，相邻格子无缝拼接
    filled=True,
    get_fill_color='properties.color',
    opacity=0.9,
)

deck_h3 = pdk.Deck(
    layers=[h3_layer],
    initial_view_state=pdk.ViewState(
        latitude=40.720, longitude=-73.975,
        zoom=13, pitch=0,
    ),
    map_style='https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json',
)

out_h3 = os.path.join(OUTPUT_DIR, 'map_H3.html')
deck_h3.to_html(out_h3, open_browser=True)
print('✅ H3 map done!')

In [ ]:
!conda install -c conda-forge keplergl -y

In [ ]:
from keplergl import KeplerGl
print('✅ keplergl 导入成功！')

In [ ]:
import h3
import pandas as pd
import os

BASE_DIR   = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

listings = pd.read_csv(
    os.path.join(BASE_DIR, 'listings.csv'),
    usecols=['latitude', 'longitude', 'neighbourhood_cleansed',
             'price', 'number_of_reviews', 'room_type']
).dropna(subset=['latitude', 'longitude'])

listings['h3_cell'] = listings.apply(
    lambda r: h3.latlng_to_cell(r['latitude'], r['longitude'], 10), axis=1
)

cell_df = listings.groupby('h3_cell').agg(
    count=('latitude', 'size'),
    avg_reviews=('number_of_reviews', 'mean')
).reset_index()

print(f'✅ H3 cells: {len(cell_df):,}')
print(cell_df.head())

In [ ]:
from keplergl import KeplerGl

map_k = KeplerGl(height=700)
map_k.add_data(data=cell_df, name='NYC Airbnb H3')
map_k

In [ ]:
from keplergl import KeplerGl
import os

# 你喜欢的视觉配置（青绿色系 + dark底图 + quantile色阶）
kepler_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 40.7128, "longitude": -74.006,
            "zoom": 11, "bearing": 0, "pitch": 0
        },
        "mapStyle": {"styleType": "dark"},
        "visState": {
            "layers": [{
                "id": "h3_layer",
                "type": "hexagonId",
                "config": {
                    "dataId": "NYC Airbnb H3",
                    "label": "Airbnb Density",
                    "columns": {"hex_id": "h3_cell"},
                    "isVisible": True,
                    "visConfig": {
                        "opacity": 0.73,
                        "coverage": 0.82,
                        "enable3d": False,
                        "colorRange": {
                            "name": "Teal",
                            "type": "sequential",
                            "category": "Custom",
                            "colors": [
                                "#0a0a1e", "#0d2b45",
                                "#0e4d6b", "#0e7490",
                                "#22b5a0", "#67e8d0"
                            ]
                        }
                    }
                },
                "visualChannels": {
                    "colorField": {"name": "count", "type": "integer"},
                    "colorScale": "quantile"
                }
            }],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "NYC Airbnb H3": [
                            {"name": "count", "format": None},
                            {"name": "avg_reviews", "format": None}
                        ]
                    },
                    "enabled": True
                },
                "brush": {"size": 23.7, "enabled": False}
            }
        }
    }
}

# 用配置重新创建地图
map_final = KeplerGl(height=700, config=kepler_config)
map_final.add_data(data=cell_df, name='NYC Airbnb H3')

# 保存
out = os.path.join(OUTPUT_DIR, 'kepler_map.html')
map_final.save_to_html(file_name=out, config=kepler_config)
print(f'✅ 保存完成 → {out}')

In [ ]:
# topic label → 数字 ID（Kepler.gl 用数字做 ordinal 颜色）
TOPIC_ID_MAP = {
    'Spanish-Speaking Visitors':  0,
    'Clean & Comfortable Stay':   1,
    'Great Location & Transit':   2,
    'Truly Enjoyable Experience': 3,
    'Feels Like Home':            4,
    'Practical Comfort':          5,
    'Nice & Pleasant Place':      6,
    'German-Speaking Visitors':   7,
    'French-Speaking Visitors':   8,
    'NYC City Experience':        9,
}

neigh_df['topic_id'] = neigh_df['dominant_topic_label'].map(TOPIC_ID_MAP).fillna(5).astype(int)

print(neigh_df[['neighborhood', 'dominant_topic_label', 'topic_id', 
                 'review_count', 'lat', 'lon']].head(8))
print(f'\nTopic distribution:\n{neigh_df["dominant_topic_label"].value_counts()}')

In [ ]:
kepler_topic_config = {
    "version": "v1",
    "config": {
        "mapState": {
            "latitude": 40.7128, "longitude": -74.006,
            "zoom": 10.5, "bearing": 0, "pitch": 0
        },
        "mapStyle": {"styleType": "dark"},
        "visState": {
            "layers": [{
                "id": "topic_points",
                "type": "point",
                "config": {
                    "dataId": "NYC Neighborhoods Topic",
                    "label": "Topic Distribution",
                    "columns": {"lat": "lat", "lng": "lon", "altitude": None},
                    "isVisible": True,
                    "visConfig": {
                        "radius": 20,
                        "fixedRadius": False,
                        "opacity": 0.9,
                        "outline": False,
                        "filled": True,
                        "radiusRange": [18, 88],
                        "colorRange": {
                            "name": "Custom Ordinal",
                            "type": "qualitative",
                            "category": "Custom",
                            "colors": [
                                "#0D0887",  # 0 Spanish-Speaking
                                "#6A00A8",  # 1 Clean & Comfortable
                                "#B12A90",  # 2 Great Location
                                "#E16462",  # 3 Enjoyable Experience
                                "#FCA636",  # 4 Feels Like Home
                                "#F0F921",  # 5 Practical Comfort
                                
                            ]
                        }
                    }
                },
                "visualChannels": {
                    "colorField": {"name": "topic_id", "type": "integer"},
                    "colorScale": "ordinal",
                    "sizeField": {"name": "review_count", "type": "integer"},
                    "sizeScale": "sqrt"
                }
            }],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "NYC Neighborhoods Topic": [
                            {"name": "neighborhood", "format": None},
                            {"name": "dominant_topic_label", "format": None},
                            {"name": "review_count", "format": None},
                            {"name": "top_keywords", "format": None}
                        ]
                    },
                    "enabled": True
                },
                "brush": {"size": 0.5, "enabled": False}
            }
        }
    }
}

map_topic2 = KeplerGl(height=700, config=kepler_topic_config)
map_topic2.add_data(data=neigh_df, name='NYC Neighborhoods Topic')
map_topic2.save_to_html(
    file_name=os.path.join(OUTPUT_DIR, 'kepler_topic_map.html'),
    config=kepler_topic_config
)
print('✅ kepler_topic_map.html 保存完成！')